# Planner

> An MPC planner with communication.

In [ ]:
#| default_exp planners.mpc

In [ ]:
#| export
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch
from loguru import logger

## JEPA Planner

In [ ]:
#| export
class JEPAGoalPlanner:
    def __init__(self, model, action_dim= 4, horizon=3, history_size=3,
                 pop_size=300, topk=30, opt_steps=30, device='cpu'):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.history_size = history_size
        self.pop_size = pop_size
        self.topk = topk
        self.opt_steps = opt_steps
        self.device = device

    def _sample_actions(self, probs):
        # probs: (B, horizon, action_dim) -> samples: (B, S, horizon, action_dim) one-hot
        B, H, A = probs.shape
        flat = probs.unsqueeze(1).expand(B, self.pop_size, H, A).reshape(-1, A)
        idx = torch.multinomial(flat, 1).view(B, self.pop_size, H)
        return idx.float()
        #return F.one_hot(idx, num_classes=A).float()

    def _update_dist(self, cost, samples):
        # cost: (B, S), samples: (B, S, horizon) -- raw action indices (dimensionless)
        B, S, H = samples.shape
        A = self.action_dim  # must come from self, since samples no longer carries it
        new_probs = torch.zeros(B, H, A, device=cost.device)
        for b in range(B):
            elite_idx = torch.topk(-cost[b], self.topk).indices
            elites = samples[b, elite_idx]  # (topk, H) -- raw indices
            for t in range(H):
                counts = torch.bincount(elites[:, t].long(), minlength=A).float()
                new_probs[b, t] = (counts + 1e-6) / (counts.sum() + 1e-6 * A)
        return new_probs

    @torch.no_grad()
    def plan(self, info_dict):
        """
        info_dict must contain (per JEPA.get_cost / rollout requirements):
          - pixels: (B, history_size, C, H, W)   recent observation history
          - action: (B, history_size, action_dim) actions taken during that history
          - goal:   (B, C, H, W)                  goal observation
          - msg_indices (optional): (B, 1, 49)    message valid only for the first
                                                    predicted step (per model.rollout)
        Returns the first action of the best plan per batch element, plus the full plan.
        """
        device = self.device
        B = info_dict["pixels"].size(0)
        probs = torch.full((B, self.horizon, self.action_dim), 1.0 / self.action_dim, device=device)
        print("Starting planning with {} optimization steps...".format(self.opt_steps))
        for _ in range(self.opt_steps):
            samples = self._sample_actions(probs)  # (B, S, horizon, action_dim)

            # full action sequence fed to rollout = known history actions + sampled future actions
            hist_action = info_dict["action"].unsqueeze(1).expand(
                B, self.pop_size, *info_dict["action"].shape[1:]
            )
            action_candidates = torch.cat([hist_action, samples], dim=2).long().to(device)

            # expand the rest of info_dict over the sample (S) dimension
            cand_info = {
                k: (v.unsqueeze(1).expand(B, self.pop_size, *v.shape[1:]).to(device)
                    if torch.is_tensor(v) else v)
                for k, v in info_dict.items()
            }
            
            cost = self.model.get_cost(cand_info, action_candidates)  # (B, S)
            probs = self._update_dist(cost, samples)
            print("Optimization step completed.")
        
        plan = torch.argmax(probs, dim=-1)       # (B, horizon)
        first_action = plan[:, 0]
        return first_action, plan
    
    @torch.no_grad()
    def eval_plan(self, info_dict, device, plan):
        
        final_action_seq = torch.cat(
        [info_dict["action"],plan], #F.one_hot(plan, num_classes=self.action_dim).float()],
        dim=1,
        ).unsqueeze(1).to(device)  # (B, 1, H+horizon, action_dim)

        final_info = {
            k: (v.unsqueeze(1).to(device) if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        final_cost = self.model.get_cost(final_info, final_action_seq).squeeze(1)  # (B,)

        # first_action = plan[:, 0]
        return final_cost
        

## Improved CEM Solver

In [ ]:
#| export
class DiscreteCEMPlanner:
    """
    CEM-based planner for a JEPA-style dynamics model whose action encoder
    consumes raw integer indices (nn.Embedding), not one-hot vectors.

    `batch_size` mirrors `CEMSolver.batch_size`: it's a memory-bounding knob
    for the CEM computation itself (candidate sampling + cost evaluation),
    completely separate from how many episodes/envs the CALLER is driving.
    The caller (evaluator/World) always passes the FULL batch; this planner
    internally sub-chunks it for the optimization, exactly like
    `CEMSolver.solve`'s `for start_idx in range(0, total_envs, self.batch_size)`
    loop -- so callers no longer need to pre-chunk episodes themselves.
    """

    def __init__(self, model, action_dim=4, horizon=3, history_size=3,
                 pop_size=300, topk=30, opt_steps=30,
                 smoothing=0.0, alpha=0.0, batch_size=None,
                 device='cpu', seed=0):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.history_size = history_size
        self.pop_size = pop_size
        self.topk = topk
        self.opt_steps = opt_steps
        self.smoothing = smoothing
        self.alpha = alpha
        # None => treat the whole incoming batch as a single chunk (same
        # behavior as before this change). Set to an int to bound memory,
        # exactly like CEMSolver.batch_size.
        self.batch_size = batch_size
        self.device = device
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)
        

In [ ]:
#| export
@patch
def _sample_actions(self: DiscreteCEMPlanner, probs):
    """
    Gumbel-max sample of categorical action indices, with the first
    sample forced to the current distribution's argmax (elitism anchor).

    probs: (B, horizon, action_dim)
    returns: indices (B, pop_size, horizon), long
    """
    B, H, A = probs.shape
    log_probs = probs.clamp_min(1e-10).log()
    log_probs = log_probs.unsqueeze(1).expand(B, self.pop_size, H, A)

    u = torch.rand(
        log_probs.shape, generator=self.torch_gen,
        device=self.device, dtype=probs.dtype,
    ).clamp_min(1e-10)
    gumbel = -(-u.log()).log()

    indices = (log_probs + gumbel).argmax(dim=-1)  # (B, S, H)
    indices[:, 0] = probs.argmax(dim=-1)            # elitism anchor
    return indices


In [ ]:
#| export
@patch
def _update_dist(self: DiscreteCEMPlanner, cost, samples):
    """
    Vectorized elite refit (no Python loops over batch or horizon).

    cost: (B, S) -- lower is better
    samples: (B, S, H) -- raw action indices, long
    returns: new_probs (B, H, action_dim)
    """
    B, S, H = samples.shape
    A = self.action_dim

    _, topk_inds = torch.topk(cost, k=self.topk, dim=1, largest=False)
    batch_idx = torch.arange(B, device=cost.device).unsqueeze(1).expand(-1, self.topk)
    elites = samples[batch_idx, topk_inds]  # (B, topk, H)

    elite_onehot = F.one_hot(elites, num_classes=A).float()  # (B, topk, H, A)
    new_probs = elite_onehot.mean(dim=1)  # (B, H, A)

    if self.smoothing > 0:
        new_probs = new_probs + self.smoothing
        new_probs = new_probs / new_probs.sum(dim=-1, keepdim=True)

    return new_probs

In [ ]:
#| export
@patch
def _plan_chunk(self: DiscreteCEMPlanner, chunk_info, verbose=False, chunk_label=""):
    """Run the full CEM optimization for one already-sized chunk."""
    cur_bs = chunk_info["pixels"].size(0)
    probs = torch.full(
        (cur_bs, self.horizon, self.action_dim), 1.0 / self.action_dim, device=self.device
    )

    cand_info = {
        k: (v.unsqueeze(1).expand(cur_bs, self.pop_size, *v.shape[1:]).to(self.device)
            if torch.is_tensor(v) else v)
        for k, v in chunk_info.items()
    }
    hist_action = chunk_info["action"].unsqueeze(1).expand(
        cur_bs, self.pop_size, *chunk_info["action"].shape[1:]
    ).to(self.device)

    for step in range(self.opt_steps):
        logger.info(f"[DiscreteCEMPlanner]{chunk_label} step {step + 1}/{self.opt_steps} -- "
                    f"mean probs: {probs.mean(dim=(0,1)).cpu().numpy()}")
        
        samples = self._sample_actions(probs)  # (cur_bs, S, horizon), long
        action_candidates = torch.cat([hist_action, samples], dim=2).long()

        cost = self.model.get_cost(cand_info, action_candidates)  # (cur_bs, S)
        new_probs = self._update_dist(cost, samples)

        probs = (
            self.alpha * probs + (1 - self.alpha) * new_probs
            if self.alpha > 0 else new_probs
        )

        if verbose:
            logger.info(
                f"[DiscreteCEMPlanner]{chunk_label} step {step + 1}/{self.opt_steps} "
                f"mean elite cost: {cost.mean().item():.4f}"
            )

    return probs


In [ ]:
#| export
@patch
@torch.no_grad()
def plan(self: DiscreteCEMPlanner, info_dict, verbose=False):
    """
    info_dict must contain (per JEPA.get_cost / rollout requirements):
        - pixels: (B, history_size, C, H, W)   recent observation history
        - action: (B, history_size)            raw action indices for history
        - goal:   (B, C, H, W)                  goal observation
        - msg_indices (optional): (B, 1, 49)

    Callers pass the FULL batch B (e.g. all episodes being evaluated
    together, mirroring World.num_envs) -- this method handles
    memory-bounding internally via self.batch_size, so callers never
    need to pre-chunk.

    Returns the first action of the best plan per batch element, plus the
    full plan.
    """
    B = info_dict["pixels"].size(0)
    chunk_size = self.batch_size or B

    probs_full = torch.empty(B, self.horizon, self.action_dim, device=self.device)

    for start_idx in range(0, B, chunk_size):
        logger.info(f"Planning for chunk {start_idx}:{min(start_idx + chunk_size, B)}...")
        end_idx = min(start_idx + chunk_size, B)
        chunk_info = {
            k: (v[start_idx:end_idx] if torch.is_tensor(v) else v)
            for k, v in info_dict.items()
        }
        probs_full[start_idx:end_idx] = self._plan_chunk(
            chunk_info, verbose=verbose, chunk_label=f" chunk[{start_idx}:{end_idx}]"
        )

    plan = torch.argmax(probs_full, dim=-1)  # (B, horizon)
    first_action = plan[:, 0]
    logger.info(f"Planning completed. Returning first actions and full plans.")
    logger.info(f"First actions: {first_action.cpu().numpy()}")
    logger.info(f"Full plans: {plan.cpu().numpy()}")
    return first_action, plan


In [ ]:
#| export
@patch
@torch.no_grad()
def eval_plan(self: DiscreteCEMPlanner, info_dict, device, plan):
    final_action_seq = torch.cat(
        [info_dict["action"], plan], dim=1
    ).unsqueeze(1).to(device).long()  # (B, 1, H+horizon)

    final_info = {
        k: (v.unsqueeze(1).to(device) if torch.is_tensor(v) else v)
        for k, v in info_dict.items()
    }
    final_cost = self.model.get_cost(final_info, final_action_seq).squeeze(1)  # (B,)
    return final_cost
